# proxy
> Caddy reverse proxy, CrowdSec security, swag containers and Cloudflare tunnel support

In [ ]:
#| default_exp proxy

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from fastcore.all import L, store_attr, listify, joins, Path
from dockeasy.core import *

## Caddyfile builder

In [ ]:
#| export
class Caddyfile(L):
	'Fluent builder for production-ready Caddyfiles'
	def __init__(self, domain, app='app', port=5001):
		store_attr()
		super().__init__([])

	def _new(self, items):
		c = type(self)(self.domain, self.app, self.port)
		c.items = items
		return c

	def _add(self, *s): return self._new(self.items + list(s))
	def _has(self, tag): return any(t == tag for t, _ in self)
	def _get(self, tag): return L(self).filter(lambda x: x[0] == tag).itemgot(1)
	def __repr__(self): return str(self)

	def email(self, addr):
		'ACME account email (global). Caddy works without it but uses anonymous account.'
		return self._add(('g', f'email {addr}'))

	def acme_dns(self, provider, token_env=None):
		'DNS-01 TLS challenge — needed when port 80 is unavailable (e.g. behind Cloudflare tunnel)'
		token = f'{{${token_env}}}' if token_env else ''
		return self._add(('s', f'\ttls {{\n\t\tacme_dns {provider} {token}\n\t}}'))

	def cloudflared(self):
		'Use http:// prefix — required when running behind cloudflared tunnel (no public port 80/443)'
		return self._add(('f', 'cloudflared'))

	def crowdsec(self, api_url='http://crowdsec:8080', key_env='CROWDSEC_API_KEY'):
		'CrowdSec bouncer — plugin, not built-in'
		return self._add(('s', f'\tcrowdsec {{\n\t\tapi_url {api_url}\n\t\tapi_key {{${key_env}}}\n\t}}'))

	def rate_limit(self, requests='100', window='1m'):
		'Rate limiting'
		return self._add(('s', f'\trate_limit {{\n\t\tzone dynamic {{\n\t\t\tkey {{http.request.remote_ip}}\n\t\t\tevents {requests}\n\t\t\twindow {window}\n\t\t}}\n\t}}'))

	def max_body(self, size='100m'):
		'Request body limit'
		return self._add(('s', f'\trequest_body {{\n\t\tmax_size {size}\n\t}}'))

	def spa(self, static_dir='/app/static'):
		'SPA preset: static files with fallback to index.html'
		return self._add(('spa', f'\troot * {static_dir}\n\ttry_files {{path}} /index.html\n\tfile_server'))

	def encode(self):
		'Compression — Caddy does NOT enable this by default, one word is enough'
		return self._add(('s', '\tencode'))

	def log(self, output='stderr'):
		'Access logging. Caddy logs errors by default; access logs are opt-in.'
		out = f'output {output}' if output != 'stderr' else ''
		return self._add(('s', f'\tlog {{\n\t\t{out}\n\t}}' if out else '\tlog'))

	def route(self, path, app, port):
		'Path-based handle block: handle /path/* { reverse_proxy app:port }'
		return self._add(('r', (path, app, port)))

	def sqlite_router(self, db, query="SELECT host, port FROM routes WHERE domain = :domain"):
		'Dynamic subdomain routing via SQLite DB (requires caddy-sqlite-router plugin)'
		return self._add(('sqlite', (db, query)))

	def __str__(self):
		'Render full Caddyfile'
		parts = []
		g = list(self._get('g'))
		if g: parts.append('{\n\t' + '\n\t'.join(g) + '\n}')
		prefix = 'http://' if self._has('f') else ''
		site = [f'{prefix}{self.domain} {{'] + listify(self._get('s'))
		if self._has('spa'): site += listify(self._get('spa'))
		elif self._has('sqlite'):
			db, query = listify(self._get('sqlite'))[0]
			site.append(f'\troute {{\n\t\tsqlite_router {db} "{query}"\n\t\treverse_proxy {{http.vars.backend_upstream}}\n\t}}')
		elif self._has('r'):
			for path, app, port in self._get('r'):
				site.append(f'\thandle {path} {{\n\t\treverse_proxy {app}:{port}\n\t}}')
			site.append(f'\thandle {{\n\t\treverse_proxy {self.app}:{self.port}\n\t}}')
		else: site.append(f'\treverse_proxy {self.app}:{self.port}')
		site.append('}')
		parts.append('\n'.join(site))
		return '\n'.join(parts)

	def save(self, path='Caddyfile'):
		'Write to disk'
		Path(path).write_text(str(self))
		return self

## Caddyfile generation

`caddyfile()` generates the Caddyfile text. `caddy()` writes it and returns service kwargs for `Compose.svc()`.

In [ ]:
#| export
def caddy(domain,  # domain to serve (e.g. example.com or sub.example.com)
          app='app',  # upstream app name (must match Compose service name)
          port=5001,  # upstream app port
          email=None,  # ACME account email (optional but good practice)
          dns=None,  # DNS-01 provider name for TLS when port 80 is blocked (e.g. 'cloudflare' or 'duckdns')
          dns_token_env=None,  # env var name for DNS API token (default: {DNS}_API_TOKEN)
          crowdsec=False,  # enable CrowdSec bouncer plugin
          crowdsec_url='http://crowdsec:8080',  # CrowdSec API URL
          cloudflared=False,  # prefix with http:// for Cloudflare tunnel setups
          encode=False,  # enable compression
          access_log=False,  # enable access logging to stderr
          routes=None,  # dict {'/path/*': ('svc', port)} for path-based multi-service routing
          )->Caddyfile:
	'Minimal Caddyfile for reverse-proxying app:port from domain. Optional ACME email, DNS-01, CrowdSec, Cloudflare tunnel, and multi-service path routing.'
	cf = Caddyfile(domain, app, port)
	if email: cf = cf.email(email)
	if dns and not cloudflared: cf = cf.acme_dns(dns, dns_token_env or f'{dns.upper()}_API_TOKEN')
	if cloudflared: cf = cf.cloudflared()
	if crowdsec: cf = cf.crowdsec(api_url=crowdsec_url)
	if encode: cf = cf.encode()
	if access_log: cf = cf.log()
	if routes:
		for path, (svc, p) in routes.items(): cf = cf.route(path, svc, p)
	return cf

def caddy_api(domain,
              app='app',
              port=5001,
              email=None,
              dns=None,
              dns_token_env=None,
              cloudflared=False,
              crowdsec=False,
              crowdsec_url='http://crowdsec:8080',
              max_body='50m',
              rate='200',
              rate_window='1m',
              routes=None,  # dict {'/path/*': ('svc', port)} for path-based multi-service routing
              ) -> str:
	'Caddyfile preset for API services — adds rate limiting and body size cap'
	return caddy(domain, app, port, email=email, dns=dns, dns_token_env=dns_token_env,
                 cloudflared=cloudflared, crowdsec=crowdsec, crowdsec_url=crowdsec_url,
                 routes=routes).rate_limit(rate, rate_window).max_body(max_body).encode()

In [ ]:
# Minimal
cf = str(caddy('myapp.example.com', port=5001))
assert 'myapp.example.com {' in cf
assert 'reverse_proxy app:5001' in cf
assert cf.count('{') == 1
print(cf)

myapp.example.com {
	reverse_proxy app:5001
}


In [ ]:
# With Cloudflare DNS
cf = str(caddy('myapp.example.com', port=5001, dns='cloudflare', email='me@example.com'))
assert 'email me@example.com' in cf
assert 'acme_dns cloudflare {$CLOUDFLARE_API_TOKEN}' in cf
print(cf)

{
	email me@example.com
}
myapp.example.com {
	tls {
		acme_dns cloudflare {$CLOUDFLARE_API_TOKEN}
	}
	reverse_proxy app:5001
}


In [ ]:
# With CrowdSec
cf = str(caddy('myapp.example.com', port=5001, crowdsec=True))
assert 'api_url http://crowdsec:8080' in cf
assert 'crowdsec' in cf
assert 'api_key {$CROWDSEC_API_KEY}' in cf
print(cf)

myapp.example.com {
	crowdsec {
		api_url http://crowdsec:8080
		api_key {$CROWDSEC_API_KEY}
	}
	reverse_proxy app:5001
}


In [ ]:
# Cloudflared mode: HTTP prefix
cf = str(caddy('myapp.example.com', port=5001, cloudflared=True))
assert cf.startswith('http://myapp.example.com {')
assert 'reverse_proxy app:5001' in cf
print(cf)

http://myapp.example.com {
	reverse_proxy app:5001
}


In [ ]:
# caddy_api()
cf = str(caddy_api('myapp.example.com'))
assert 'rate_limit' in cf and 'max_size 50m' in cf and 'encode' in cf
assert 'reverse_proxy app:5001' in cf
print('caddy_api() OK')

cf = str(caddy_api('myapp.example.com', crowdsec=True, max_body='100m', rate='50', rate_window='30s'))
assert 'crowdsec' in cf and 'events 50' in cf and 'window 30s' in cf and 'max_size 100m' in cf
print('caddy_api() with crowdsec OK')

# Caddyfile.spa()
cf = str(Caddyfile('myapp.example.com').spa())
assert 'try_files {path} /index.html' in cf and 'file_server' in cf
assert 'reverse_proxy' not in cf
print('spa() OK')

# Caddyfile.encode() and .log()
cf = str(caddy('myapp.example.com', encode=True, access_log=True))
assert '\tencode' in cf and '\tlog' in cf
print('encode() and log() OK')

cf = str(Caddyfile('myapp.example.com').log(output='/var/log/caddy/access.log'))
assert 'output /var/log/caddy/access.log' in cf
print('log(output=...) OK')

caddy_api() OK
caddy_api() with crowdsec OK
spa() OK
encode() and log() OK
log(output=...) OK


In [ ]:
# Multi-service routing: FastHTML app + UCall JSON-RPC
cf = caddy('myapp.example.com', routes={'/rpc/*': ('ucall', 8545)})
assert 'handle /rpc/*' in str(cf)
assert 'reverse_proxy ucall:8545' in str(cf)
assert 'handle {' in str(cf)
assert 'reverse_proxy app:5001' in str(cf)
assert 'reverse_proxy app:5001' not in str(cf).split('handle {')[0]  # fallback is last
print(cf)

# route() fluent API — same result
cf2 = Caddyfile('myapp.example.com').route('/rpc/*', 'ucall', 8545)
assert str(cf) == str(cf2)

# routes + crowdsec: middleware before handle blocks
cf3 = caddy('myapp.example.com', crowdsec=True, routes={'/rpc/*': ('ucall', 8545)})
s = str(cf3)
assert s.index('crowdsec') < s.index('handle /rpc/*')
print('multi-route OK')

## Services

In [ ]:
#| export
def _caddy_img(crowdsec, cloudflare):
    if crowdsec and cloudflare: return 'ghcr.io/buildplan/csdp-caddy:latest'
    elif crowdsec: return 'serfriz/caddy-crowdsec:latest'
    elif cloudflare: return 'serfriz/caddy-cloudflare:latest'
    else: return 'caddy:2'

def caddy_svc(domain, app='app', port=5001, *, dns=None, email=None,
              crowdsec=False, cloudflared=False, conf='Caddyfile',
              routes=None,  # dict {'/path/*': ('svc', port)} — adds services to depends_on
              **kw):
    'Write Caddyfile and return Caddy service kwargs for Compose.svc()'
    caddy(domain, app, port, email=email, dns=dns, cloudflared=cloudflared,
          crowdsec=crowdsec, routes=routes).save(conf)
    is_cf = dns == 'cloudflare'
    tok = f'{dns.upper()}_API_TOKEN' if dns else None
    env = {}
    if dns: env[tok] = '${%s}' % tok
    if crowdsec: env['CROWDSEC_API_KEY'] = '${CROWDSEC_API_KEY}'
    conf_path = Path(conf)
    vol_key = str(conf_path.resolve()) if conf_path.is_absolute() else f'./{conf_path.name}'
    deps = [app] + [svc for svc, _ in (routes or {}).values()]
    return service(
        image=kw.pop('image', _caddy_img(crowdsec, is_cf)),
        ports=None if cloudflared else ['80:80', '443:443', '443:443/udp'],
        env=env or None,
        volumes={vol_key: '/etc/caddy/Caddyfile', 'caddy_data': '/data', 'caddy_config': '/config'},
        depends_on=deps, networks=['web'], restart='unless-stopped', **kw
    )

In [ ]:
import os, time, tempfile, subprocess
from fastcore.all import urlread, Path

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    kw = caddy_svc('ex.com', conf=f'{tmp}/Caddyfile')
    assert kw['image'] == 'caddy:2'
    assert kw['ports'] == ['80:80', '443:443', '443:443/udp']
    assert kw['depends_on'] == ['app']
    print('caddy_svc() basic OK')

with tempfile.TemporaryDirectory() as tmp:
    kw = caddy_svc('ex.com', cloudflared=True, conf=f'{tmp}/Caddyfile')
    assert 'ports' not in kw
    assert kw['image'] == 'caddy:2'
    assert Path(f'{tmp}/Caddyfile').read_text().startswith('http://ex.com {')
    print('caddy_svc() cloudflared OK')

with tempfile.TemporaryDirectory() as tmp:
    kw = caddy_svc('ex.com', crowdsec=True, conf=f'{tmp}/Caddyfile')
    assert kw['image'] == 'serfriz/caddy-crowdsec:latest'
    assert 'CROWDSEC_API_KEY=${CROWDSEC_API_KEY}' in kw['environment']
    print('caddy_svc() crowdsec OK')

with tempfile.TemporaryDirectory() as tmp:
    kw = caddy_svc('ex.com', crowdsec=True, cloudflared=True, conf=f'{tmp}/Caddyfile')
    assert kw['image'] == 'serfriz/caddy-crowdsec:latest'
    assert 'ports' not in kw
    print('caddy_svc() crowdsec+cloudflared OK')

with tempfile.TemporaryDirectory() as tmp:
    kw = caddy_svc('ex.com', dns='cloudflare', conf=f'{tmp}/Caddyfile')
    assert kw['image'] == 'serfriz/caddy-cloudflare:latest'
    assert 'CLOUDFLARE_API_TOKEN=${CLOUDFLARE_API_TOKEN}' in kw['environment']
    print('caddy_svc() cloudflare-dns OK')

with tempfile.TemporaryDirectory() as tmp:
    kw = caddy_svc('ex.com', dns='cloudflare', crowdsec=True, conf=f'{tmp}/Caddyfile')
    assert kw['image'] == 'ghcr.io/buildplan/csdp-caddy:latest'
    assert 'CLOUDFLARE_API_TOKEN=${CLOUDFLARE_API_TOKEN}' in kw['environment']
    assert 'CROWDSEC_API_KEY=${CROWDSEC_API_KEY}' in kw['environment']
    print('caddy_svc() cloudflare-dns+crowdsec OK')

with tempfile.TemporaryDirectory() as tmp:
    kw = caddy_svc('ex.duckdns.org', dns='duckdns', conf=f'{tmp}/Caddyfile', image='serfriz/caddy-duckdns:latest')
    assert kw['image'] == 'serfriz/caddy-duckdns:latest'
    assert 'DUCKDNS_API_TOKEN=${DUCKDNS_API_TOKEN}' in kw['environment']
    cf = Path(f'{tmp}/Caddyfile').read_text()
    assert 'acme_dns duckdns {$DUCKDNS_API_TOKEN}' in cf
    print('caddy_svc() duckdns OK')

caddy_svc() basic OK
caddy_svc() cloudflared OK
caddy_svc() crowdsec OK
caddy_svc() crowdsec+cloudflared OK
caddy_svc() cloudflare-dns OK
caddy_svc() cloudflare-dns+crowdsec OK
caddy_svc() duckdns OK


In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    kw = caddy_svc('ex.com', routes={'/rpc/*': ('ucall', 8545)}, conf=f'{tmp}/Caddyfile')
    assert set(kw['depends_on']) == {'app', 'ucall'}
    cf = Path(f'{tmp}/Caddyfile').read_text()
    assert 'handle /rpc/*' in cf and 'reverse_proxy ucall:8545' in cf
    assert 'handle {' in cf and 'reverse_proxy app:5001' in cf
    print('caddy_svc() multi-route OK')

with tempfile.TemporaryDirectory() as tmp:
    kw = caddy_svc('ex.com', routes={'/rpc/*': ('ucall', 8545), '/admin/*': ('admin', 9000)},
                   conf=f'{tmp}/Caddyfile')
    assert set(kw['depends_on']) == {'app', 'ucall', 'admin'}
    print('caddy_svc() multi-route depends_on OK')

In [ ]:
#| export
def cloudflared_svc(token_env='${CF_TUNNEL_TOKEN}', url=None, **kw):
    'Cloudflare tunnel service kwargs for Compose.svc(). Pass url to set ingress inline (e.g. url="http://caddy").'
    cmd = f'tunnel --no-autoupdate run{" --url " + url if url else ""}'
    return service(image='cloudflare/cloudflared:latest', command=cmd,
                   env={'TUNNEL_TOKEN': token_env}, networks=['web'], restart='unless-stopped', **kw)

In [ ]:
kw = cloudflared_svc()
assert kw['image'] == 'cloudflare/cloudflared:latest'
assert kw['command'] == 'tunnel --no-autoupdate run'
assert 'TUNNEL_TOKEN=${CF_TUNNEL_TOKEN}' in kw['environment']
print('cloudflared_svc() OK')

kw2 = cloudflared_svc(url='http://caddy')
assert kw2['command'] == 'tunnel --no-autoupdate run --url http://caddy'
print('cloudflared_svc() with url OK')

cloudflared_svc() OK
cloudflared_svc() with url OK


In [ ]:
#| export
def crowdsec(collections=None, bouncer_key_env='CROWDSEC_BOUNCER_KEY', **kw):
    'CrowdSec agent service kwargs for Compose.svc()'
    cols = joins(' ', listify(collections or ['crowdsecurity/linux', 'crowdsecurity/caddy', 'crowdsecurity/http-cve']))
    return service(
        image='crowdsecurity/crowdsec:latest',
        env={'COLLECTIONS': cols, 'BOUNCER_KEY_caddy': f'${{{bouncer_key_env}}}'},
        volumes={'crowdsec-db': '/var/lib/crowdsec/data', 'crowdsec-config': '/etc/crowdsec'},
        networks=['web'], restart='unless-stopped', **kw
    )

In [ ]:
kw = crowdsec()
assert kw['image'] == 'crowdsecurity/crowdsec:latest'
assert any('crowdsecurity/caddy' in e for e in kw['environment'])
assert 'BOUNCER_KEY_caddy=${CROWDSEC_BOUNCER_KEY}' in kw['environment']
assert any('crowdsec-db' in v for v in kw['volumes'])
print('crowdsec() OK')

kw2 = crowdsec(collections=['crowdsecurity/linux', 'crowdsecurity/nginx'])
assert any('crowdsecurity/nginx' in e for e in kw2['environment'])
print('crowdsec() custom collections OK')

crowdsec() OK
crowdsec() custom collections OK


## SQLite Dynamic Router

[`caddy-sqlite-router`](https://github.com/AnswerDotAI/caddy-sqlite-router) resolves `*.domain` subdomains to backends by querying a SQLite DB — ideal for multi-tenant apps where routes change at runtime. Requires a custom Caddy build; `caddy_sqlite_dockerfile()` generates the xcaddy Dockerfile.

In [ ]:
#| export
def caddy_sqlite_dockerfile():
    'Multi-stage Dockerfile: Caddy + caddy-sqlite-router plugin (requires CGO via xcaddy)'
    return (Dockerfile()
            .from_('caddy:2-builder', as_='builder')
            .run('apk add --no-cache gcc musl-dev')
            .run('CGO_ENABLED=1 xcaddy build --with github.com/AnswerDotAI/caddy-sqlite-router')
            .from_('caddy:2-alpine')
            .copy('/usr/bin/caddy', '/usr/bin/caddy', from_='builder'))

def caddy_sqlite_svc(domain, db='/data/routes.db', image='caddy-sqlite:latest', *,
                     query="SELECT host, port FROM routes WHERE domain = :domain",
                     email=None, crowdsec=False, conf='Caddyfile', **kw):
    'Caddy service with SQLite dynamic subdomain routing. Build image with caddy_sqlite_dockerfile(). db is the in-container path; mount it via **kw volumes.'
    cf = Caddyfile(domain)
    if email: cf = cf.email(email)
    if crowdsec: cf = cf.crowdsec()
    cf.sqlite_router(db, query).save(conf)
    conf_path = Path(conf)
    vol_key = str(conf_path.resolve()) if conf_path.is_absolute() else f'./{conf_path.name}'
    env = {'CROWDSEC_API_KEY': '${CROWDSEC_API_KEY}'} if crowdsec else None
    return service(
        image=image, ports=['80:80', '443:443', '443:443/udp'],
        env=env,
        volumes={vol_key: '/etc/caddy/Caddyfile', 'caddy_data': '/data', 'caddy_config': '/config'},
        networks=['web'], restart='unless-stopped', **kw
    )

In [ ]:
df = caddy_sqlite_dockerfile()
s = str(df)
assert 'xcaddy build' in s and 'caddy-sqlite-router' in s and 'CGO_ENABLED=1' in s
assert 'FROM caddy:2-builder AS builder' in s
assert 'FROM caddy:2-alpine' in s
assert '--from=builder' in s
print(df)

with tempfile.TemporaryDirectory() as tmp:
    cf_path = f'{tmp}/Caddyfile'
    kw = caddy_sqlite_svc('*.example.com', db='/data/routes.db', conf=cf_path)
    assert kw['image'] == 'caddy-sqlite:latest'
    assert kw['ports'] == ['80:80', '443:443', '443:443/udp']
    cf = Path(cf_path).read_text()
    assert 'sqlite_router /data/routes.db' in cf
    assert '{http.vars.backend_upstream}' in cf
    print('caddy_sqlite_svc() OK')
    print(cf)

# custom query
with tempfile.TemporaryDirectory() as tmp:
    cf_path = f'{tmp}/Caddyfile'
    caddy_sqlite_svc('*.example.com', db='tenants.db',
                     query='SELECT host, port FROM tenants WHERE subdomain = :domain',
                     conf=cf_path)
    cf = Path(cf_path).read_text()
    assert 'tenants.db' in cf and 'subdomain' in cf
    print('caddy_sqlite_svc() custom query OK')

In [ ]:
# Fluent builder: sqlite_router renders correct directive block
cf = Caddyfile('*.example.com').sqlite_router('/data/routes.db')
s = str(cf)
assert 'sqlite_router /data/routes.db' in s, s
assert '{http.vars.backend_upstream}' in s
assert 'route {' in s
assert s.count('reverse_proxy') == 1  # only inside route block, no bare fallback
print(cf)
print('Caddyfile.sqlite_router() OK')

# sqlite_router overrides bare reverse_proxy — app:port fallback must not appear
cf2 = Caddyfile('*.example.com', app='myapp', port=5001).sqlite_router('/data/routes.db')
assert 'reverse_proxy myapp:5001' not in str(cf2)
print('sqlite_router suppresses bare reverse_proxy OK')

# Middleware ordering: crowdsec block must precede sqlite_router block
cf3 = Caddyfile('*.example.com').crowdsec().sqlite_router('/data/routes.db')
s3 = str(cf3)
assert s3.index('crowdsec') < s3.index('sqlite_router'), 'crowdsec must appear before sqlite_router'
print('middleware ordering OK')
print(cf3)

### SQLite routing simulation

`caddy-sqlite-router` extracts the subdomain from each request and runs `SELECT host, port FROM routes WHERE domain = :domain` — `:domain` receives only the subdomain portion (e.g. `app1` from `app1.example.com`). The tests below create a real SQLite DB and verify the query contract the plugin relies on.

In [ ]:
import sqlite3

def _make_routes_db(path, rows):
    'Create a routes DB matching the default caddy-sqlite-router schema'
    con = sqlite3.connect(path)
    con.execute('CREATE TABLE routes (domain TEXT PRIMARY KEY, host TEXT NOT NULL, port INTEGER NOT NULL)')
    con.executemany('INSERT INTO routes VALUES (?,?,?)', rows)
    con.commit()
    con.close()

ROUTES = [('app1', 'app1-svc', 8001), ('app2', 'app2-svc', 8002), ('admin', 'admin-svc', 9000)]

with tempfile.TemporaryDirectory() as tmp:
    db_path = f'{tmp}/routes.db'
    _make_routes_db(db_path, ROUTES)

    query = "SELECT host, port FROM routes WHERE domain = :domain"
    con = sqlite3.connect(db_path)

    # Known routes resolve correctly
    assert con.execute(query, {'domain': 'app1'}).fetchone() == ('app1-svc', 8001)
    assert con.execute(query, {'domain': 'app2'}).fetchone() == ('app2-svc', 8002)
    assert con.execute(query, {'domain': 'admin'}).fetchone() == ('admin-svc', 9000)

    # Unknown subdomain returns no row (caddy-sqlite-router will 502/404)
    assert con.execute(query, {'domain': 'notexist'}).fetchone() is None

    # `:domain` is the subdomain only, not full hostname — 'app1.example.com' won't match 'app1'
    assert con.execute(query, {'domain': 'app1.example.com'}).fetchone() is None

    con.close()

    # Caddyfile points to the correct db path
    cf_path = f'{tmp}/Caddyfile'
    kw = caddy_sqlite_svc('*.example.com', db=db_path, conf=cf_path)
    cf_text = Path(cf_path).read_text()
    assert db_path in cf_text
    assert 'sqlite_router' in cf_text
    print(cf_text)

print('SQLite routing simulation OK')

In [ ]:
# Custom query routing — multi-tenant pattern with different table/column names
with tempfile.TemporaryDirectory() as tmp:
    db_path = f'{tmp}/tenants.db'
    con = sqlite3.connect(db_path)
    con.execute('CREATE TABLE tenants (subdomain TEXT PRIMARY KEY, host TEXT NOT NULL, port INTEGER NOT NULL)')
    con.executemany('INSERT INTO tenants VALUES (?,?,?)', [
        ('acme', 'acme-app', 3000), ('globex', 'globex-app', 3001)])
    con.commit()

    custom_query = 'SELECT host, port FROM tenants WHERE subdomain = :domain'
    assert con.execute(custom_query, {'domain': 'acme'}).fetchone() == ('acme-app', 3000)
    assert con.execute(custom_query, {'domain': 'globex'}).fetchone() == ('globex-app', 3001)
    assert con.execute(custom_query, {'domain': 'unknown'}).fetchone() is None
    con.close()

    cf_path = f'{tmp}/Caddyfile'
    caddy_sqlite_svc('*.example.com', db=db_path, query=custom_query, conf=cf_path)
    cf_text = Path(cf_path).read_text()
    assert 'subdomain' in cf_text and 'tenants' in cf_text
    print(cf_text)

print('custom query routing OK')

## Integration test: SQLite router — two FastHTML apps

Live test — builds a custom Caddy image with `caddy-sqlite-router`, spins up two FastHTML apps, populates a routes DB, and verifies that `app1.DOMAIN` and `app2.DOMAIN` are routed to the correct backends.

**Prerequisites:**
- `BASE_DOMAIN` env var: your base domain (e.g. `angalama.com`)
- `CF_TUNNEL_TOKEN` env var: from Cloudflare Zero Trust → Tunnels
- Wildcard tunnel ingress rule: `*.BASE_DOMAIN → http://caddy`
- Wildcard DNS CNAME in Cloudflare: `*.BASE_DOMAIN → <tunnel-id>.cfargotunnel.com`

In [ ]:
#| eval: false
BASE_DOMAIN = 'angalama.com'
# CF_TUNNEL_TOKEN read from host env by docker-compose: ${CF_TUNNEL_TOKEN}

# ── two minimal FastHTML apps with distinct responses ─────────────────────────
def _make_app_dir(name):
    d = Path(tempfile.mkdtemp(dir='.'))
    (d/'app.py').write_text(f'''\
from fasthtml.common import *
app, rt = fast_app()

@rt('/')
def get(): return Titled('{name}', P('routed to {name} ✓'))

serve()
''')
    (d/'pyproject.toml').write_text('''\
[project]
name = "app"
version = "0.1.0"
dependencies = ["python-fasthtml", "starlette<0.46"]
''')
    fasthtml_app().save(d/'Dockerfile')
    return d

app1_dir = _make_app_dir('app1')
app2_dir = _make_app_dir('app2')

# ── routes DB: subdomain → (host, port) ──────────────────────────────────────
# caddy-sqlite-router extracts the subdomain only: app1.angalama.com → domain='app1'
inf_dir = Path(tempfile.mkdtemp(dir='.'))
db_host_path = str(inf_dir/'routes.db')
con = sqlite3.connect(db_host_path)
con.execute('CREATE TABLE routes (domain TEXT PRIMARY KEY, host TEXT NOT NULL, port INTEGER NOT NULL)')
con.executemany('INSERT INTO routes VALUES (?,?,?)', [
    ('app1', 'app1', 5001),
    ('app2', 'app2', 5001),
])
con.commit(); con.close()

# ── caddy-sqlite image: saved to temp dir, built by compose ──────────────────
caddy_dir = Path(tempfile.mkdtemp(dir='.'))
caddy_sqlite_dockerfile().save(caddy_dir/'Dockerfile')

# ── compose stack ─────────────────────────────────────────────────────────────
cf_path = str(inf_dir/'Caddyfile')
dc_path = str(inf_dir/'docker-compose.yml')

caddy_kw = caddy_sqlite_svc(f'*.{BASE_DOMAIN}', db='/routes.db',
                             build=str(caddy_dir), conf=cf_path)
caddy_kw['volumes'].append(f'{db_host_path}:/routes.db')  # bind-mount routes DB into caddy

dc = (Compose()
    .svc('app1', build=str(app1_dir), networks=['web'], restart='unless-stopped')
    .svc('app2', build=str(app2_dir), networks=['web'], restart='unless-stopped')
    .svc('caddy', **caddy_kw)
    .svc('cloudflared', **cloudflared_svc())
    .network('web').volume('caddy_data').volume('caddy_config'))

print(dc)
print('--- Caddyfile ---')
print(Path(cf_path).read_text())

# ── run & verify subdomain routing ───────────────────────────────────────────
try:
    dc.up(path=dc_path)
    print('Waiting for image build + tunnel...')
    results = {}
    for sub in ('app1', 'app2'):
        url = f'https://{sub}.{BASE_DOMAIN}'
        for i in range(18):
            time.sleep(10)
            try: results[sub] = urlread(url); break
            except Exception as e: print(f'  [{(i+1)*10}s] {sub} not ready: {e}')

    print('=== container logs ===')
    print(dc.logs(path=dc_path))

    assert 'app1' in results, 'app1 never responded'
    assert 'routed to app1' in results['app1'], f'app1 wrong response: {results["app1"][:200]}'
    assert 'app2' in results, 'app2 never responded'
    assert 'routed to app2' in results['app2'], f'app2 wrong response: {results["app2"][:200]}'
    print(f'✓ app1.{BASE_DOMAIN} → app1 (routed via SQLite)')
    print(f'✓ app2.{BASE_DOMAIN} → app2 (routed via SQLite)')
finally:
    dc.down(path=dc_path, v=True, remove_orphans=True)
    print('Cleaned up.')

## Example: FastHTML app with Caddy

Minimal stacks — run any with `dc.save('docker-compose.yml')` then `docker compose up -d`.

In [ ]:
tmp = tempfile.mkdtemp()

# Stack A: Direct (Caddy auto-TLS, ports 80+443 open)
dc = (Compose()
    .svc('app', build='.', networks=['web'], restart='unless-stopped')
    .svc('caddy', **caddy_svc('myapp.example.com', port=5001, conf=f'{tmp}/Caddyfile'))
    .network('web', driver='bridge').volume('caddy_data').volume('caddy_config'))

d = dc.to_dict()
assert d['services']['caddy']['image'] == 'caddy:2'
assert '80:80' in d['services']['caddy']['ports']
print('=== Stack A: Direct (Caddy auto-TLS) ===')
print(dc)

=== Stack A: Direct (Caddy auto-TLS) ===
services:
  app:
    build: .
    networks:
    - web
    restart: unless-stopped
  caddy:
    image: caddy:2
    depends_on:
    - app
    ports:
    - 80:80
    - 443:443
    - 443:443/udp
    volumes:
    - /private/var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmp2pn1i7ra/Caddyfile:/etc/caddy/Caddyfile
    - caddy_data:/data
    - caddy_config:/config
    networks:
    - web
    restart: unless-stopped
networks:
  web:
    driver: bridge
volumes:
  caddy_data: null
  caddy_config: null



In [ ]:
tmp = tempfile.mkdtemp()
# Stack B: cloudflared tunnel (zero open ports)
dc = (Compose()
    .svc('app', build='.', networks=['web'], restart='unless-stopped')
    .svc('caddy', **caddy_svc('myapp.example.com', port=5001, cloudflared=True, conf=f'{tmp}/Caddyfile'))
    .svc('cloudflared', **cloudflared_svc())
    .network('web').volume('caddy_data').volume('caddy_config'))

d = dc.to_dict()
assert 'ports' not in d['services']['caddy']
assert d['services']['cloudflared']['image'] == 'cloudflare/cloudflared:latest'
print('=== Stack B: Cloudflared (zero open ports) ===')
print(dc)

=== Stack B: Cloudflared (zero open ports) ===
services:
  app:
    build: .
    networks:
    - web
    restart: unless-stopped
  caddy:
    image: caddy:2
    depends_on:
    - app
    volumes:
    - /private/var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmprvwce3po/Caddyfile:/etc/caddy/Caddyfile
    - caddy_data:/data
    - caddy_config:/config
    networks:
    - web
    restart: unless-stopped
  cloudflared:
    image: cloudflare/cloudflared:latest
    command: tunnel --no-autoupdate run
    environment:
    - TUNNEL_TOKEN=${CF_TUNNEL_TOKEN}
    networks:
    - web
    restart: unless-stopped
networks:
  web: null
volumes:
  caddy_data: null
  caddy_config: null



In [ ]:
tmp = tempfile.mkdtemp()
# Stack C: CrowdSec + cloudflared (full security, zero open ports)
dc = (Compose()
    .svc('app', build='.', networks=['web'], restart='unless-stopped')
    .svc('caddy', **caddy_svc('myapp.example.com', port=5001, crowdsec=True, cloudflared=True, conf=f'{tmp}/Caddyfile'))
    .svc('crowdsec', **crowdsec())
    .svc('cloudflared', **cloudflared_svc())
    .network('web')
    .volume('caddy_data').volume('caddy_config')
    .volume('crowdsec-db').volume('crowdsec-config'))

d = dc.to_dict()
assert d['services']['caddy']['image'] == 'serfriz/caddy-crowdsec:latest'
assert d['services']['crowdsec']['image'] == 'crowdsecurity/crowdsec:latest'
assert 'ports' not in d['services']['caddy']
print('=== Stack C: CrowdSec + cloudflared (zero open ports) ===')
print(dc)

=== Stack C: CrowdSec + cloudflared (zero open ports) ===
services:
  app:
    build: .
    networks:
    - web
    restart: unless-stopped
  caddy:
    image: serfriz/caddy-crowdsec:latest
    depends_on:
    - app
    volumes:
    - /private/var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmp7hi9ftvr/Caddyfile:/etc/caddy/Caddyfile
    - caddy_data:/data
    - caddy_config:/config
    environment:
    - CROWDSEC_API_KEY=${CROWDSEC_API_KEY}
    networks:
    - web
    restart: unless-stopped
  crowdsec:
    image: crowdsecurity/crowdsec:latest
    volumes:
    - crowdsec-db:/var/lib/crowdsec/data
    - crowdsec-config:/etc/crowdsec
    environment:
    - COLLECTIONS=crowdsecurity/linux crowdsecurity/caddy crowdsecurity/http-cve
    - BOUNCER_KEY_caddy=${CROWDSEC_BOUNCER_KEY}
    networks:
    - web
    restart: unless-stopped
  cloudflared:
    image: cloudflare/cloudflared:latest
    command: tunnel --no-autoupdate run
    environment:
    - TUNNEL_TOKEN=${CF_TUNNEL_TOKEN}
    network

In [ ]:
tmp = tempfile.mkdtemp()

# Stack D: Cloudflare DNS-01 (wildcard cert via ACME, ports 80+443 open, no tunnel)
dc = (Compose()
    .svc('app', build='.', networks=['web'], restart='unless-stopped')
    .svc('caddy', **caddy_svc('myapp.example.com', port=5001, dns='cloudflare', email='me@example.com', conf=f'{tmp}/Caddyfile'))
    .network('web').volume('caddy_data').volume('caddy_config'))

d = dc.to_dict()
assert d['services']['caddy']['image'] == 'serfriz/caddy-cloudflare:latest'
assert '80:80' in d['services']['caddy']['ports']
assert 'CLOUDFLARE_API_TOKEN=${CLOUDFLARE_API_TOKEN}' in d['services']['caddy']['environment']
cf = Path(f'{tmp}/Caddyfile').read_text()
assert 'acme_dns cloudflare {$CLOUDFLARE_API_TOKEN}' in cf
print('=== Stack D: Cloudflare DNS-01 (wildcard cert, direct) ===')
print(dc)
print('--- Caddyfile ---')
print(cf)

=== Stack D: Cloudflare DNS-01 (wildcard cert, direct) ===
services:
  app:
    build: .
    networks:
    - web
    restart: unless-stopped
  caddy:
    image: serfriz/caddy-cloudflare:latest
    depends_on:
    - app
    ports:
    - 80:80
    - 443:443
    - 443:443/udp
    volumes:
    - /private/var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmptw1nzplx/Caddyfile:/etc/caddy/Caddyfile
    - caddy_data:/data
    - caddy_config:/config
    environment:
    - CLOUDFLARE_API_TOKEN=${CLOUDFLARE_API_TOKEN}
    networks:
    - web
    restart: unless-stopped
networks:
  web: null
volumes:
  caddy_data: null
  caddy_config: null

--- Caddyfile ---
{
	email me@example.com
}
myapp.example.com {
	tls {
		acme_dns cloudflare {$CLOUDFLARE_API_TOKEN}
	}
	reverse_proxy app:5001
}


In [ ]:
tmp = tempfile.mkdtemp()

# Stack E: Cloudflare DNS + CrowdSec (full security, direct ports open)
dc = (Compose()
    .svc('app', build='.', networks=['web'], restart='unless-stopped')
    .svc('caddy', **caddy_svc('myapp.example.com', port=5001,
                              dns='cloudflare', crowdsec=True,
                              email='me@example.com', conf=f'{tmp}/Caddyfile'))
    .svc('crowdsec', **crowdsec())
    .network('web')
    .volume('caddy_data').volume('caddy_config')
    .volume('crowdsec-db').volume('crowdsec-config'))

d = dc.to_dict()
assert d['services']['caddy']['image'] == 'ghcr.io/buildplan/csdp-caddy:latest'
assert '80:80' in d['services']['caddy']['ports']
assert d['services']['crowdsec']['image'] == 'crowdsecurity/crowdsec:latest'
assert 'CLOUDFLARE_API_TOKEN=${CLOUDFLARE_API_TOKEN}' in d['services']['caddy']['environment']
assert 'CROWDSEC_API_KEY=${CROWDSEC_API_KEY}' in d['services']['caddy']['environment']
cf = Path(f'{tmp}/Caddyfile').read_text()
assert 'acme_dns cloudflare {$CLOUDFLARE_API_TOKEN}' in cf
assert 'crowdsec' in cf
print('=== Stack E: Cloudflare DNS + CrowdSec (full security, direct ports) ===')
print(dc)
print('--- Caddyfile ---')
print(cf)

=== Stack E: Cloudflare DNS + CrowdSec (full security, direct ports) ===
services:
  app:
    build: .
    networks:
    - web
    restart: unless-stopped
  caddy:
    image: ghcr.io/buildplan/csdp-caddy:latest
    depends_on:
    - app
    ports:
    - 80:80
    - 443:443
    - 443:443/udp
    volumes:
    - /private/var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmp4y8f06cw/Caddyfile:/etc/caddy/Caddyfile
    - caddy_data:/data
    - caddy_config:/config
    environment:
    - CLOUDFLARE_API_TOKEN=${CLOUDFLARE_API_TOKEN}
    - CROWDSEC_API_KEY=${CROWDSEC_API_KEY}
    networks:
    - web
    restart: unless-stopped
  crowdsec:
    image: crowdsecurity/crowdsec:latest
    volumes:
    - crowdsec-db:/var/lib/crowdsec/data
    - crowdsec-config:/etc/crowdsec
    environment:
    - COLLECTIONS=crowdsecurity/linux crowdsecurity/caddy crowdsecurity/http-cve
    - BOUNCER_KEY_caddy=${CROWDSEC_BOUNCER_KEY}
    networks:
    - web
    restart: unless-stopped
networks:
  web: null
volumes:
  ca

### Stack F: FastHTML + UCall JSON-RPC (multi-service routing)

Path-based routing: `/rpc/*` → UCall (port 8545), everything else → FastHTML app (port 5001). Both services are auto-added to `depends_on`.

In [ ]:
tmp = tempfile.mkdtemp()

# Stack F: FastHTML app + UCall JSON-RPC — path-based routing under one domain
dc = (Compose()
    .svc('app', build='.', networks=['web'], restart='unless-stopped')
    .svc('ucall', image='unum-cloud/ucall:latest', networks=['web'], restart='unless-stopped')
    .svc('caddy', **caddy_svc('myapp.example.com', port=5001,
                              routes={'/rpc/*': ('ucall', 8545)},
                              conf=f'{tmp}/Caddyfile'))
    .network('web').volume('caddy_data').volume('caddy_config'))

d = dc.to_dict()
assert set(d['services']['caddy']['depends_on']) == {'app', 'ucall'}
cf = Path(f'{tmp}/Caddyfile').read_text()
assert 'handle /rpc/*' in cf and 'reverse_proxy ucall:8545' in cf
assert 'handle {' in cf and 'reverse_proxy app:5001' in cf
print('=== Stack F: FastHTML + UCall JSON-RPC (multi-service) ===')
print(dc)
print('--- Caddyfile ---')
print(cf)

## Integration test: FastHTML + Caddy + cloudflared

Live test — spins up a real stack and verifies the app is reachable over the internet.

**Prerequisites:**
- `DOMAIN` env var: hostname matching your Cloudflare tunnel ingress rule (e.g. `myapp.example.com`)
- `CF_TUNNEL_TOKEN` env var: from Cloudflare Zero Trust → Tunnels
- Tunnel ingress rule configured in Cloudflare dashboard: `DOMAIN → http://caddy`

In [ ]:
#| eval: false
DOMAIN = 'fastops.angalama.com'
# CF_TUNNEL_TOKEN read from host env by docker-compose: ${CF_TUNNEL_TOKEN}

# ── app ───────────────────────────────────────────────────────────────────────
app_dir = Path(tempfile.mkdtemp(dir='.'))
(app_dir / 'app.py').write_text('''\
from fasthtml.common import *
app, rt = fast_app()

@rt('/')
def get(): return Titled('dockeasy-proxy-test', P('Caddy + cloudflared ✓'))

serve()
''')
(app_dir / 'pyproject.toml').write_text('''\
[project]
name = "proxy-test"
version = "0.1.0"
dependencies = ["python-fasthtml", "starlette<0.46"]
''')
fasthtml_app().save(app_dir / 'Dockerfile')

tag = 'dockeasy-proxy-test:latest'

# ── compose stack ─────────────────────────────────────────────────────────────
inf_dir = Path(tempfile.mkdtemp(dir='.'))
cf_path = str(inf_dir / 'Caddyfile')
dc_path = str(inf_dir / 'docker-compose.yml')

dc = (Compose()
    .svc('app', build=str(app_dir), image=tag, networks=['web'], restart='unless-stopped')
    .svc('caddy', **caddy_svc(DOMAIN, cloudflared=True, conf=cf_path))
    .svc('cloudflared', **cloudflared_svc(url='http://caddy'))
    .network('web').volume('caddy_data').volume('caddy_config'))

print(dc)
print('--- Caddyfile ---')
print(Path(cf_path).read_text())

# ── run & verify ──────────────────────────────────────────────────────────────
try:
    dc.up(path=dc_path)
    print('Waiting for cloudflared tunnel to connect...')
    html = None
    for i in range(12):
        time.sleep(10)
        try: html = urlread(f'https://{DOMAIN}'); break
        except Exception as e: print(f'  [{(i+1)*10}s] not ready: {e}')
    print('=== container logs ===')
    print(dc.logs(path=dc_path))
    assert html and 'dockeasy-proxy-test' in html, f'Unexpected response: {html[:200] if html else "no response"}'
    print(f'✓ App reachable at https://{DOMAIN}')
finally:
    dc.down(path=dc_path, v=True, remove_orphans=True)
    rmi(tag, force=True)
    print('Cleaned up.')

services:
  app:
    image: dockeasy-proxy-test:latest
    build: /Users/71293/code/dockeasy/nbs/tmpode386ks
    networks:
    - web
    restart: unless-stopped
  caddy:
    image: caddy:2
    depends_on:
    - app
    volumes:
    - /Users/71293/code/dockeasy/nbs/tmp881ik_hr/Caddyfile:/etc/caddy/Caddyfile
    - caddy_data:/data
    - caddy_config:/config
    networks:
    - web
    restart: unless-stopped
  cloudflared:
    image: cloudflare/cloudflared:latest
    command: tunnel --no-autoupdate run --url http://caddy
    environment:
    - TUNNEL_TOKEN=${CF_TUNNEL_TOKEN}
    networks:
    - web
    restart: unless-stopped
networks:
  web: null
volumes:
  caddy_data: null
  caddy_config: null

--- Caddyfile ---
http://fastops.angalama.com {
	reverse_proxy app:5001
}
Waiting for cloudflared tunnel to connect...
=== container logs ===
caddy-1        | {"level":"info","ts":1774238518.5918498,"msg":"maxprocs: Leaving GOMAXPROCS=2: CPU quota undefined"}
cloudflared-1  | 2026-03-23T04:01:58

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()